Exploration Charts — Zero-Shot NLI (10 custom labels) fork
==============================================================
Charts the Zero-Shot NLI (10 custom labels) scores produced by `05.1_song_analysis_zeroshot.ipynb`.

The companion notebook `06.2_exploration_charts_goemotions.ipynb` runs the identical chart set over the other
classifier. Both forks are produced by `04_classification.ipynb` under one **shared
scoring contract** (see its header), so they differ only where they must:

| | zero-shot NLI (04 §4A → 05.1 → 06.1) | GoEmotions (04 §4B → 05.2 → 06.2) |
|---|---|---|
| scoring | independent per-label, [0, 1] | *same* |
| `unclassified` | no scoreable lyrics | *same* — drops the same ~107 songs |
| confidence | flagged at 0.30, never dropped | *same* |
| **labels** | **10, hand-picked for songs** | **28, fixed (Reddit-trained)** |
| **includes `neutral`** | **no** | **yes** |
| **model** | **`bart-large-mnli`, zero-shot** | **`roberta-base-go_emotions`, supervised** |
| **typical top score** | **~0.97** | **~0.50** |

The bold rows are the real, irreducible differences; everything above them used
to differ too, purely by config. The last row still matters for reading charts:
RoBERTa's sigmoids are calibrated systematically lower than bart-mnli's
entailment probabilities, so **a raw 0.4 does not mean the same thing in each
fork** even under the shared contract.

Absolute-magnitude charts below are therefore labelled fork-local. The **z-score
charts (§ 4) are the ones that survive a cross-fork read** — standardising
within a fork cancels the calibration offset and leaves only "which regions are
unusually high on this emotion, relative to this fork's own baseline".

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

# ── Fork config ───────────────────────────────────────────────────────────────
LABEL        = "Zero-Shot NLI (10 custom labels)"
INPUT_PATH   = PROCESSED / "05.1_titles_emotion_scores_zeroshot.csv"
DOMINANT_COL = "dominant_emotion"   # column charted as "the song's emotion"
EXCLUDE      = []          # labels held out of the emotion charts
TOP_N        = 10            # emotions charted; all 10 labels fit

df = pd.read_csv(INPUT_PATH)

emotion_cols = [
    c for c in df.columns
    if c.startswith("emotion_") and c.replace("emotion_", "") not in EXCLUDE
]
# Charted subset: the strongest TOP_N emotions overall. With 28 labels a full
# grid is unreadable, so the tail is trimmed here rather than in every chart.
charted_cols = (
    df[emotion_cols].mean().sort_values(ascending=False).head(TOP_N).index.tolist()
)
charted_names = [c.replace("emotion_", "") for c in charted_cols]

print(f"{LABEL}: {len(df)} songs, {len(emotion_cols)} emotion columns "
      f"(excluding {EXCLUDE or 'none'}), charting top {len(charted_cols)}")
print(charted_names)
display(df.head())

Zero-Shot NLI (10 custom labels): 1471 songs, 10 emotion columns (excluding none), charting top 10
['longing', 'sensual', 'lonely', 'love', 'heartbreak', 'grief', 'anger', 'despair', 'hope', 'joy']


,rank,artist,title,region,spotify_uri,dominant_emotion,dominant_score,low_confidence,emotion_love,emotion_longing,emotion_joy,emotion_heartbreak,emotion_grief,emotion_despair,emotion_hope,emotion_lonely,emotion_sensual,emotion_anger
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,sensual,0.8266,False,0.0089,0.4467,0.0070,0.5252,0.3905,0.1312,0.0922,0.3994,0.8266,0.3854
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,lonely,0.9945,False,0.8727,0.9616,0.0528,0.5929,0.2682,0.6702,0.7960,0.9945,0.9174,0.6488
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,sensual,0.9812,False,0.1551,0.7746,0.0950,0.0028,0.0043,0.0064,0.0919,0.1293,0.9812,0.0778
3,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,longing,0.9732,False,0.3941,0.9732,0.0171,0.5878,0.2483,0.2008,0.1080,0.7073,0.7838,0.3424
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,6iOndD4OFo7GkaDypWQIou,love,0.9922,False,0.9922,0.9760,0.2966,0.0018,0.0018,0.0016,0.1824,0.0086,0.9816,0.0055


In [2]:
# ── Palette ───────────────────────────────────────────────────────────────────
# Sequential = ONE hue light→dark (magnitude). Diverging = two poles + a neutral
# gray midpoint (polarity around zero). Never a rainbow ramp for either job — a
# rainbow makes non-adjacent values look adjacent and hides the actual ordering.
SEQ_BLUE = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
    "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
    "#184f95", "#104281", "#0d366b",
]
DIVERGING = [[0.0, "#0d366b"], [0.25, "#3987e5"], [0.5, "#f0efec"],
             [0.75, "#e34948"], [1.0, "#7d1f1e"]]

# Shared z-score domain for § 4 / § 4b, hard-coded rather than taken from this
# fork's own max. Both forks' regional z-scores peak near ±2.4, so one fixed
# limit is what actually makes the two notebooks' z-charts stackable — a
# per-fork max would quietly give the same SD value a different colour and a
# different bar position in each. Widened past the data to leave label room.
Z_LIM = 2.9

# Fixed categorical order — hues are assigned by slot and never cycled or
# re-ordered by rank, so a given region keeps its colour across every chart.
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
               "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

regions = sorted(df["region"].dropna().unique())
if len(regions) > len(CATEGORICAL):
    raise ValueError(
        f"{len(regions)} regions but only {len(CATEGORICAL)} validated hues. "
        "Fold the smallest into 'Other' or facet instead of generating a 9th hue."
    )
REGION_COLOR = dict(zip(regions, CATEGORICAL))

# `Global` is a reference playlist, not a market. § 4 / § 4b hold it out of the
# baseline and then score it against that baseline, so its position reads as
# "how Global sits relative to the markets" instead of it being averaged into
# the very mean it is compared with. Set to None to standardise across all
# regions instead.
REF_REGION = "Global" if "Global" in regions else None
baseline_regions = [r for r in regions if r != REF_REGION]
print(f"z baseline: {len(baseline_regions)} regions"
      + (f", holding out {REF_REGION!r}" if REF_REGION else " (no region held out)"))

LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, -apple-system, Helvetica, sans-serif", size=13,
              color="#0b0b0b"),
    title_font_size=17,
    margin=dict(l=90, r=40, t=90, b=70),
)
print(f"{len(regions)} regions: {regions}")

z baseline: 7 regions, holding out 'Global'
8 regions: ['Argentina', 'Colombia', 'Global', 'Japan', 'Singapore', 'Spain', 'Taiwan', 'USA']


### 1. Coverage and confidence

Only songs with no scoreable lyrics were dropped upstream in the 05.x join, and
both forks drop the same ~107 of them — so **this fork and its sibling are
charting the identical song set**, and per-region counts are directly
comparable between the two notebooks.

Low-confidence songs are *kept*, flagged rather than dropped. The flag uses the
same 0.30 bar in both forks, but because the two models are calibrated
differently the flagged *share* will not match — that difference is a real
property of the classifiers, which is exactly why it's shown here instead of
being silently absorbed into the row count.

In [3]:
region_counts = df["region"].value_counts().reindex(regions)

fig = go.Figure(go.Bar(
    x=region_counts.index,
    y=region_counts.values,
    marker=dict(color=[REGION_COLOR[r] for r in region_counts.index],
                cornerradius=4),
    text=region_counts.values,
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y} songs<extra></extra>",
    showlegend=False,
))
fig.update_layout(
    title=f"Classified songs per region — {LABEL}<br>"
          f"<sup>{len(df)} songs total; unclassified already removed in 05.x</sup>",
    yaxis_title="songs", xaxis_title=None, bargap=0.35,
    height=420, width=900, **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False)
fig.show()

# Confidence is carried as data, so report it rather than let it hide in the counts.
conf = (
    df.groupby("region")
      .agg(songs=("spotify_uri", "size"),
           median_dominant_score=("dominant_score", "median"),
           pct_low_confidence=("low_confidence", lambda s: s.mean() * 100))
      .reindex(regions)
)
print(f"Overall low_confidence share: {df['low_confidence'].mean():.1%} "
      f"(bar = 0.30, shared with the other fork)")
display(conf.round(2))

Overall low_confidence share: 0.3% (bar = 0.30, shared with the other fork)


,songs,median_dominant_score,pct_low_confidence
region,,,
Argentina,178,0.98,0.00
Colombia,188,0.97,0.00
Global,196,0.98,0.00
Japan,188,0.95,1.06
Singapore,183,0.98,0.00
Spain,193,0.98,1.04
Taiwan,149,0.96,0.00
USA,196,0.98,0.51


### 2. Dominant emotion mix

Which label wins outright per song, counted overall and split by region. Counts
are fork-local — the label sets don't line up, so a bar here has no counterpart
in the sibling notebook.

In [4]:
dom = df[DOMINANT_COL].value_counts()
dom = dom[~dom.index.isin(EXCLUDE)]

fig = go.Figure(go.Bar(
    x=dom.values, y=dom.index, orientation="h",
    marker=dict(color="#2a78d6", cornerradius=4),
    text=dom.values, textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x} songs<extra></extra>",
))
fig.update_layout(
    title=f"Dominant emotion across all songs — {LABEL}<br>"
          f"<sup>column charted: <code>{DOMINANT_COL}</code></sup>",
    xaxis_title="songs", yaxis=dict(autorange="reversed"),
    height=max(360, 26 * len(dom) + 140), width=860, bargap=0.35, **LAYOUT,
)
fig.update_xaxes(gridcolor="#eceae5", zeroline=False)
fig.update_yaxes(showgrid=False)
fig.show()

# Share of each region's songs, so a large region doesn't dominate by size alone
mix = (
    pd.crosstab(df["region"], df[DOMINANT_COL], normalize="index")
      .reindex(regions)
      .drop(columns=[c for c in EXCLUDE if c in df[DOMINANT_COL].unique()], errors="ignore")
)
mix = mix[dom.index[: min(12, len(dom))]]
display((mix * 100).round(1))

dominant_emotion,longing,sensual,love,heartbreak,lonely,anger,despair,hope,grief,joy
region,,,,,,,,,,
Argentina,38.2,24.2,9.0,12.4,10.1,4.5,0.6,1.1,0.0,0.0
Colombia,32.4,29.3,9.6,11.7,8.5,5.9,1.1,0.0,0.0,1.6
Global,35.2,19.9,11.7,15.3,10.2,4.1,1.5,1.0,0.5,0.5
Japan,35.6,12.8,24.5,6.4,11.7,5.3,0.0,1.6,0.5,1.6
Singapore,39.9,18.0,13.1,13.7,9.8,2.2,1.6,1.1,0.5,0.0
Spain,39.9,27.5,9.8,8.8,8.3,4.7,0.0,0.0,1.0,0.0
Taiwan,36.9,24.8,8.7,8.7,9.4,5.4,2.0,2.7,0.7,0.7
USA,31.6,16.3,12.2,14.3,15.8,6.6,1.5,0.0,1.5,0.0


### 3. Regional emotion profile — raw mean scores *(fork-local)*

Mean score per region per emotion, on this fork's native scale. Useful for
reading *within* the grid (which emotion runs hottest in a region), **not** for
comparing a cell against the sibling notebook's same-named cell.

In [5]:
heat = df.groupby("region")[charted_cols].mean().reindex(regions)
heat.columns = charted_names

fig = px.imshow(
    heat.T,
    labels=dict(x="", y="", color="mean score"),
    color_continuous_scale=SEQ_BLUE,
    aspect="auto",
    text_auto=".2f",
)
fig.update_traces(
    textfont_size=11,
    hovertemplate="<b>%{x}</b> · %{y}<br>mean score %{z:.3f}<extra></extra>",
    xgap=2, ygap=2,   # surface gap between cells
)
fig.update_layout(
    title=f"Mean emotion score by region — {LABEL}<br>"
          f"<sup>Fork-local scale. Compare cells within this grid, not against 06.x's other fork.</sup>",
    height=60 + 34 * len(charted_names) + 140, width=1000,
    coloraxis_colorbar=dict(title="mean", thickness=12, len=0.7),
    **LAYOUT,
)
fig.show()
display(heat.round(3))

,longing,sensual,lonely,love,heartbreak,grief,anger,despair,hope,joy
region,,,,,,,,,,
Argentina,0.853,0.680,0.606,0.542,0.557,0.455,0.392,0.364,0.214,0.122
Colombia,0.788,0.720,0.539,0.568,0.465,0.373,0.407,0.301,0.216,0.152
Global,0.873,0.692,0.596,0.593,0.548,0.462,0.319,0.352,0.281,0.169
Japan,0.781,0.605,0.503,0.607,0.422,0.326,0.168,0.211,0.315,0.191
Singapore,0.869,0.703,0.607,0.643,0.551,0.462,0.303,0.343,0.326,0.201
Spain,0.848,0.761,0.557,0.565,0.484,0.401,0.414,0.292,0.221,0.136
Taiwan,0.806,0.720,0.615,0.587,0.501,0.448,0.377,0.355,0.356,0.280
USA,0.850,0.671,0.626,0.537,0.584,0.495,0.360,0.415,0.241,0.140


### 4. Regional character — z-scored within this fork *(cross-fork readable)*

Each emotion column is standardised inside this fork as
`(region mean − baseline mean) / baseline std`, where the baseline is every
region **except `Global`**. Global is then scored against that baseline rather
than folded into it, so it works as a genuine reference point — "how the Global
playlist sits relative to the markets" — instead of being part of the mean it is
measured against. As a result the *markets* centre on zero; Global floats.

Standardising strips out both the overall scale and each label's own baseline
popularity, leaving a pure "how unusual is this region on this emotion" reading.

This is the chart to hold next to the sibling notebook's version of it. A value
of +1.5 means the same thing in both — *1.5 standard deviations above this
classifier's own market baseline* — even though the underlying raw scores are
on incomparable scales. Diverging blue↔red with a gray midpoint, because zero is
a real and meaningful centre here.

In [6]:
region_means = df.groupby("region")[charted_cols].mean().reindex(regions)

# Baseline = the market regions only. REF_REGION is scored against it, not part
# of it, so it can be read as a reference line rather than a ninth data point
# dragging the mean toward itself.
base_means = region_means.loc[baseline_regions]
z = (region_means - base_means.mean()) / base_means.std(ddof=0)
z.columns = charted_names
z = z.fillna(0)

# The colour domain is Z_LIM, not this fork's own max: a per-fork max would give
# the two notebooks different scales for the same SD value, which defeats the
# whole point of standardising. Guard rather than silently clip.
if float(np.abs(z.values).max()) > Z_LIM:
    raise ValueError(
        f"z reaches {np.abs(z.values).max():.2f} SD, past Z_LIM={Z_LIM}. "
        "Raise Z_LIM in BOTH forks so they stay on one scale."
    )

fig = px.imshow(
    z.T,
    labels=dict(x="", y="", color="z-score"),
    color_continuous_scale=DIVERGING,
    zmin=-Z_LIM, zmax=Z_LIM,
    aspect="auto",
    text_auto=".1f",
)
fig.update_traces(
    textfont_size=11,
    hovertemplate="<b>%{x}</b> · %{y}<br>%{z:+.2f} SD vs this fork's regional mean<extra></extra>",
    xgap=2, ygap=2,
)
fig.update_layout(
    title=f"Regional emotion character — {LABEL}<br>"
          f"<sup>Standard deviations from this fork's own regional mean. "
          f"Red = distinctively high, blue = distinctively low. Comparable across forks.</sup>",
    height=60 + 34 * len(charted_names) + 150, width=1000,
    coloraxis_colorbar=dict(title="SD", thickness=12, len=0.7),
    **LAYOUT,
)
fig.show()
display(z.round(2))

,longing,sensual,lonely,love,heartbreak,grief,anger,despair,hope,joy
region,,,,,,,,,,
Argentina,0.75,-0.31,0.63,-1.05,0.90,0.60,0.57,0.64,-1.00,-1.04
Colombia,-1.22,0.57,-0.94,-0.29,-0.83,-0.92,0.76,-0.41,-0.96,-0.44
Global,1.39,-0.06,0.40,0.41,0.74,0.71,-0.34,0.43,0.19,-0.12
Japan,-1.44,-1.96,-1.77,0.84,-1.63,-1.78,-2.21,-1.90,0.80,0.32
Singapore,1.26,0.19,0.65,1.86,0.79,0.72,-0.54,0.29,1.00,0.52
Spain,0.62,1.47,-0.51,-0.40,-0.47,-0.40,0.85,-0.57,-0.88,-0.75
Taiwan,-0.66,0.55,0.83,0.24,-0.16,0.46,0.38,0.48,1.55,2.07
USA,0.68,-0.51,1.10,-1.19,1.40,1.32,0.18,1.47,-0.51,-0.69


### 4b. The same z-scores, read by position

The grid above is a fine overview, but colour cannot be decoded to ±0.3
precision — its values are only readable because they're printed into the cells.
Here the identical matrix is encoded by **position** against a shared zero line:
one row per emotion, one dot per region, with colour carrying region *identity*
(the same fixed hues as every other chart in this notebook) instead of magnitude.

Two things this shows that the heatmap hides:

- **Row width is the finding.** z is standardised *across markets within an
  emotion*, so the market dots in every row centre on zero by construction — that
  balance carries no information, but the spread does. A wide row is an emotion
  the markets genuinely split on; a tight row is one they broadly agree about.
  Rows are sorted by that width.
- **Ordering within a row** is read straight off the axis rather than inferred
  from two similar shades.

Both this chart and § 4 above are pinned to a fixed `Z_LIM` shared with the
sibling notebook, so the two forks' versions can be put side by side and read as
one scale.

Three regions are labelled per row: the **low** and **high** market poles, and
**`Global`**, the held-out reference. Global is drawn above its dot rather than
beside it, because it lands mid-row where the horizontal space is already taken.
Note that the poles and the row sort are computed over the markets only — Global
is charted and labelled but never defines the spread, for the same reason it is
kept out of the baseline. The remaining regions are in the legend, the hover, and
the table underneath.

In [7]:
# Rows ordered by how much the *markets* disagree — the widest row goes on top.
# REF_REGION is excluded here for the same reason it is excluded from the
# baseline: it is a reference, so it should not define the spread it is read against.
spread = (z.loc[baseline_regions].max() - z.loc[baseline_regions].min()).sort_values()
row_order = spread.index.tolist()

fig = go.Figure()

# Alternating row bands, so an eye tracking a row left→right doesn't slip off it.
for i in range(0, len(row_order), 2):
    fig.add_shape(type="rect", xref="paper", yref="y", x0=0, x1=1,
                  y0=i - 0.5, y1=i + 0.5, fillcolor="#f7f6f4",
                  line_width=0, layer="below")

for region in regions:
    fig.add_trace(go.Scatter(
        x=z.loc[region, row_order].values,
        y=row_order,
        mode="markers",
        name=region,
        # 2px surface ring, not a border: it keeps overlapping dots countable.
        marker=dict(size=11, color=REGION_COLOR[region],
                    line=dict(width=2, color="white")),
        hovertemplate=f"<b>{region}</b> · %{{y}}<br>%{{x:+.2f}} SD<extra></extra>",
    ))

# Three direct labels per row — the two market poles and the reference region.
# Everyone else is carried by the legend, the hover and the table below; a label
# on all eight would be unreadable.
lo = z.loc[baseline_regions, row_order].idxmin()
hi = z.loc[baseline_regions, row_order].idxmax()

# Poles: outside the dot, on the side the dot points to, so they never sit on data.
fig.add_trace(go.Scatter(
    x=[z.loc[lo[e], e] for e in row_order] + [z.loc[hi[e], e] for e in row_order],
    y=row_order + row_order,
    mode="text",
    text=[lo[e] + "  " for e in row_order] + ["  " + hi[e] for e in row_order],
    textposition=["middle left"] * len(row_order) + ["middle right"] * len(row_order),
    textfont=dict(size=11, color="#6b6862"),
    showlegend=False, hoverinfo="skip", cliponaxis=False,
))

# Reference region: its own lane above the dots, as annotations rather than
# scatter text. Scatter's "top center" offsets by only the marker radius, so
# where the reference sits on top of a pole the two labels collide; a fixed
# pixel yshift clears the pole labels in every row regardless of the values.
if REF_REGION:
    for e in row_order:
        fig.add_annotation(
            x=z.loc[REF_REGION, e], y=e, text=REF_REGION,
            showarrow=False, yshift=17, xanchor="center",
            font=dict(size=10, color="#6b6862"),
        )

fig.update_layout(
    title=f"Regional character per emotion — {LABEL}<br>"
          f"<sup>Each dot is one region's z-score; 0 = this fork's own market baseline"
          + (f", which excludes {REF_REGION}.<br>" if REF_REGION else ".<br>")
          + f"Axis fixed at ±{Z_LIM} SD, so this chart stacks directly against the other fork's. "
            f"Rows sorted by how much the markets disagree.</sup>",
    # The zero rule is the axis zeroline, not an added shape: a shape at x=0 draws
    # alongside the x=0 gridline and renders as a doubled line.
    xaxis=dict(range=[-Z_LIM, Z_LIM], gridcolor="#eceae5", dtick=1,
               zeroline=True, zerolinecolor="#8a8681", zerolinewidth=1.5,
               title=dict(text="standard deviations from this fork's market baseline",
                          font=dict(size=12))),
    yaxis=dict(categoryorder="array", categoryarray=row_order,
               showgrid=False, title=None),
    height=56 * len(row_order) + 210, width=1000,
    legend=dict(orientation="h", y=-0.14, title=None),
    **LAYOUT,
)
fig.show()

# Table twin: the three labelled values per row, plus the width the chart sorts on.
display(
    pd.DataFrame({
        "market spread (SD)": spread,
        "lowest": lo,
        "lowest z": [z.loc[lo[e], e] for e in row_order],
        "highest": hi,
        "highest z": [z.loc[hi[e], e] for e in row_order],
        f"{REF_REGION} z": [z.loc[REF_REGION, e] for e in row_order]
                           if REF_REGION else None,
    }).sort_values("market spread (SD)", ascending=False).round(2)
)

,market spread (SD),lowest,lowest z,highest,highest z,Global z
sensual,3.42,Japan,-1.96,Spain,1.47,-0.06
despair,3.38,Japan,-1.90,USA,1.47,0.43
joy,3.12,Argentina,-1.04,Taiwan,2.07,-0.12
grief,3.10,Japan,-1.78,USA,1.32,0.71
anger,3.06,Japan,-2.21,Spain,0.85,-0.34
love,3.05,USA,-1.19,Singapore,1.86,0.41
heartbreak,3.03,Japan,-1.63,USA,1.40,0.74
lonely,2.87,Japan,-1.77,USA,1.10,0.40
longing,2.70,Japan,-1.44,Singapore,1.26,1.39
hope,2.55,Argentina,-1.00,Taiwan,1.55,0.19


### 5. Where each emotion peaks

Top 5 regions per emotion. Read the *ordering* of the bars; the heights are on
the fork-local scale again.

In [8]:
n = len(charted_cols)
ncols = 3
nrows = -(-n // ncols)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[c.replace("emotion_", "") for c in charted_cols],
    vertical_spacing=0.09 if nrows <= 4 else 0.05,
    horizontal_spacing=0.07,
)

for idx, col in enumerate(charted_cols):
    r, c = idx // ncols + 1, idx % ncols + 1
    top = df.groupby("region")[col].mean().sort_values(ascending=False).head(5)
    fig.add_trace(
        go.Bar(
            x=top.index, y=top.values,
            marker=dict(color=[REGION_COLOR[i] for i in top.index], cornerradius=4),
            hovertemplate="<b>%{x}</b><br>mean %{y:.3f}<extra></extra>",
            showlegend=False,
        ),
        row=r, col=c,
    )

fig.update_annotations(font_size=13)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False, title=None)
fig.update_xaxes(showgrid=False, tickangle=-35, tickfont_size=10)
fig.update_layout(
    title_text=f"Top 5 regions per emotion — {LABEL}<br>"
               f"<sup>Fork-local mean scores; region colours are fixed across every chart here.</sup>",
    height=260 * nrows + 120, width=1150, bargap=0.35, **LAYOUT,
)
fig.show()

### 6. Score spread within each region

Means hide bimodality — a region can average mid on an emotion because every
song is mid, or because half its songs are extreme. Box plots separate those.

In [9]:
melted = (
    df[["region"] + charted_cols]
      .melt(id_vars="region", var_name="emotion", value_name="score")
)
melted["emotion"] = melted["emotion"].str.replace("emotion_", "", regex=False)

fig = px.box(
    melted, x="emotion", y="score", color="region",
    color_discrete_map=REGION_COLOR,
    category_orders={"emotion": charted_names, "region": regions},
    points=False,
)
fig.update_traces(line_width=1.5, marker_size=4)
fig.update_layout(
    title=f"Score distribution by emotion and region — {LABEL}<br>"
          f"<sup>Box = IQR, line = median. Fork-local scale.</sup>",
    xaxis_title=None, yaxis_title="score",
    boxmode="group", height=560, width=max(1000, 78 * len(charted_names)),
    legend=dict(orientation="h", y=-0.28, title=None),
    **LAYOUT,
)
fig.update_yaxes(gridcolor="#eceae5", zeroline=False)
fig.update_xaxes(showgrid=False, tickangle=-30)
fig.show()

### 7. Table view

Every chart above has a numeric counterpart here — required so nothing in this
notebook is readable by colour alone.

In [10]:
summary = pd.concat(
    {
        "mean score": df.groupby("region")[charted_cols].mean().reindex(regions).T,
    },
    axis=1,
)
summary.index = charted_names
display(summary.round(3))

print("\nGlobal ranking of emotions in this fork:")
display(
    df[emotion_cols].mean().sort_values(ascending=False)
      .rename("mean score").to_frame().round(3)
)

mean score                                                      
region      Argentina Colombia Global  Japan Singapore  Spain Taiwan    USA
longing         0.853    0.788  0.873  0.781     0.869  0.848  0.806  0.850
sensual         0.680    0.720  0.692  0.605     0.703  0.761  0.720  0.671
lonely          0.606    0.539  0.596  0.503     0.607  0.557  0.615  0.626
love            0.542    0.568  0.593  0.607     0.643  0.565  0.587  0.537
heartbreak      0.557    0.465  0.548  0.422     0.551  0.484  0.501  0.584
grief           0.455    0.373  0.462  0.326     0.462  0.401  0.448  0.495
anger           0.392    0.407  0.319  0.168     0.303  0.414  0.377  0.360
despair         0.364    0.301  0.352  0.211     0.343  0.292  0.355  0.415
hope            0.214    0.216  0.281  0.315     0.326  0.221  0.356  0.241
joy             0.122    0.152  0.169  0.191     0.201  0.136  0.280  0.140


Global ranking of emotions in this fork:


,mean score
emotion_longing,0.834
emotion_sensual,0.694
emotion_lonely,0.580
emotion_love,0.580
emotion_heartbreak,0.514
emotion_grief,0.427
emotion_anger,0.341
emotion_despair,0.329
emotion_hope,0.269
emotion_joy,0.171
